# Database Overview & Data Quality Audit

A general-purpose, schema-agnostic audit of `pmc_fair.db`: how many rows are in each table,
and how much data is missing (NULL, or empty string for text columns) in each column of each
table. No figures — this is a plain-text reference report, useful for sanity-checking the
database after a pipeline run or before starting a new analysis notebook.

The notebook discovers tables and columns automatically via SQLite's own schema
(`sqlite_master` / `PRAGMA table_info`), so it doesn't need updating when tables or columns
are added or removed — it always reports on whatever the database actually contains right now.

Everything printed below is also written to `fair_db_overview_exports/database_overview.txt`
(plus the underlying data as two CSVs) so it can be read outside a notebook too.

| Section | Content |
|---|---|
| 0 | Setup |
| 1 | Table row counts |
| 2 | Missing-data audit (every column of every table) |
| 3 | Summary and exports |

---

In [1]:
import importlib.util, subprocess, sys

# A bare `pip install pandas` here resolves to whatever `pip` is first on
# $PATH, not this kernel's own interpreter -- in this environment that is
# a separate, PEP-668 "externally managed" pip that refuses to run at all,
# even though pandas is already installed in this kernel's own venv. Check
# first, and if actually needed, install via `sys.executable -m pip` so it
# always targets this kernel's environment instead of whatever's on PATH.
_required = ["pandas"]
_missing = [pkg for pkg in _required if importlib.util.find_spec(pkg) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *_missing])
else:
    print("Required packages already installed.")


Required packages already installed.


## 0 · Setup

In [2]:
import sqlite3
import io
from datetime import datetime
from pathlib import Path
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.width', 120)

DB_PATH = Path("../data/processed/pmc_fair.db")
assert DB_PATH.exists(), f"Database not found: {DB_PATH}"
FIGURES_DIR = Path("../figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
SAVEFIG_KWARGS = dict(dpi=150, bbox_inches="tight")
con = sqlite3.connect(DB_PATH)

def q(sql, params=()):
    """Run a query and return a DataFrame."""
    return pd.read_sql_query(sql, con, params=params)

# ── Report buffer: everything emit()'d goes to both stdout and the text file ─
_report = io.StringIO()

def emit(text=""):
    print(text)
    _report.write(str(text) + "\n")

emit(f"Database overview -- generated {datetime.now():%Y-%m-%d %H:%M}")
emit(f"Database path: {DB_PATH.resolve()}")
emit(f"File size: {DB_PATH.stat().st_size / 1e9:.2f} GB")
emit()

# All real tables, excluding SQLite's own internal bookkeeping table
tables = sorted(r[0] for r in con.execute(
    "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'"
).fetchall())
emit(f"Tables found: {len(tables)}")
for t in tables:
    emit(f"  - {t}")

Database overview -- generated 2026-11-08 21:55
Database path: /home/shsam/tuc/gitlab/samuel-research/10yearsfairdas/data/processed/pmc_fair.db
File size: 30.91 GB

Tables found: 28
  - accession_number
  - affiliation
  - article
  - article_author
  - article_discipline
  - article_funder
  - article_keyword
  - article_mesh
  - author
  - author_affiliation
  - code_availability
  - code_link
  - da_link
  - das_category
  - data_availability
  - ethics_statement
  - ethics_sub_type
  - fuji_article_link
  - fuji_assessment
  - fuji_metric
  - given_name_gender
  - journal
  - journal_oa
  - keyword
  - link_rot
  - link_rot_run
  - repo_classification
  - supplementary_material


---
## 1 · Table Row Counts

In [3]:
# ── 1.1 Row count per table ──────────────────────────────────────────────────
counts = []
for t in tables:
    n = con.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
    counts.append({"table": t, "rows": n})

df_counts = pd.DataFrame(counts).sort_values("rows", ascending=False).reset_index(drop=True)
df_counts["rows_fmt"] = df_counts["rows"].apply(lambda n: f"{n:,}")

emit()
emit("=" * 60)
emit("TABLE ROW COUNTS (largest first)")
emit("=" * 60)
emit(df_counts[["table", "rows_fmt"]].rename(columns={"rows_fmt": "rows"}).to_string(index=False))
emit()
emit(f"Total rows across all tables: {df_counts['rows'].sum():,}")


TABLE ROW COUNTS (largest first)
                 table       rows
        article_author 47,920,596
      accession_number 39,438,861
    author_affiliation 36,459,340
          article_mesh 36,127,961
       article_keyword 27,605,784
                author 13,394,080
           affiliation 11,308,170
        article_funder 10,634,891
               article  6,885,547
    article_discipline  6,885,547
          das_category  6,885,547
supplementary_material  6,697,313
               keyword  4,528,449
           fuji_metric  3,549,073
     data_availability  3,013,543
     given_name_gender  2,150,835
      ethics_statement  1,245,851
               da_link  1,169,559
   repo_classification  1,169,559
              link_rot  1,129,798
     fuji_article_link    253,112
       fuji_assessment    210,336
       ethics_sub_type    124,196
             code_link     86,101
     code_availability     64,157
               journal     22,138
            journal_oa     22,138
          link

---
## 2 · Missing-Data Audit

For every column of every table: how many rows have a `NULL` value (and, for text columns,
also an empty string `''` — this database consistently uses `''` rather than `NULL` to mean
"no value" in several text fields, e.g. `pub_year`). One query per table (not per column), so
this stays efficient even on the largest tables (tens of millions of rows).

In [4]:
# ── 2.1 Per-column missing-value counts, one query per table ────────────────
def missing_report_for_table(table):
    cols = con.execute(f'PRAGMA table_info("{table}")').fetchall()
    # cols: (cid, name, type, notnull, dflt_value, pk)
    select_parts = ["COUNT(*) AS __total_rows__"]
    col_names = []
    for _, name, ctype, *_ in cols:
        col_names.append(name)
        cond = f'"{name}" IS NULL'
        if ctype and ctype.upper() in ("TEXT", "VARCHAR", "CHAR", "CLOB"):
            cond += f' OR "{name}" = \'\''
        select_parts.append(f'SUM(CASE WHEN {cond} THEN 1 ELSE 0 END) AS "{name}"')
    sql = f'SELECT {", ".join(select_parts)} FROM "{table}"'
    row = con.execute(sql).fetchone()
    total = row[0]
    rows = []
    for name, n_missing in zip(col_names, row[1:]):
        n_missing = n_missing or 0
        rows.append({
            "table": table, "column": name, "total_rows": total,
            "n_missing": n_missing,
            "pct_missing": (n_missing / total * 100) if total else 0.0,
        })
    return rows

all_missing = []
for t in tables:
    emit(f"Auditing {t} ...")
    all_missing.extend(missing_report_for_table(t))

df_missing = pd.DataFrame(all_missing)

Auditing accession_number ...


Auditing affiliation ...


Auditing article ...


Auditing article_author ...


Auditing article_discipline ...


Auditing article_funder ...


... (12 outputs left out)

Auditing journal ...
Auditing journal_oa ...
Auditing keyword ...


Auditing link_rot ...


Auditing link_rot_run ...
Auditing repo_classification ...


Auditing supplementary_material ...


In [5]:
# ── 2.2 Full per-table / per-column breakdown ────────────────────────────────
emit()
emit("=" * 90)
emit("MISSING-DATA AUDIT (NULL, or '' for text columns) -- every column, every table")
emit("=" * 90)

for t in tables:
    sub = df_missing[df_missing["table"] == t].copy()
    total = sub["total_rows"].iloc[0]
    emit()
    emit(f"--- {t}  ({total:,} rows) ---")
    view = sub[["column", "n_missing", "pct_missing"]].copy()
    view["n_missing"] = view["n_missing"].apply(lambda n: f"{n:,}")
    view["pct_missing"] = view["pct_missing"].apply(lambda p: f"{p:5.1f}%")
    emit(view.to_string(index=False))


MISSING-DATA AUDIT (NULL, or '' for text columns) -- every column, every table

--- accession_number  (39,438,861 rows) ---
    column n_missing pct_missing
    acc_id         0        0.0%
article_id         0        0.0%
   db_name         0        0.0%
 accession         0        0.0%
       url 7,332,722       18.6%

--- affiliation  (11,308,170 rows) ---
        column n_missing pct_missing
affiliation_id         0        0.0%
   affiliation         0        0.0%

--- article  (6,885,547 rows) ---
             column n_missing pct_missing
         article_id         0        0.0%
         journal_id         0        0.0%
             pmc_id 6,885,547      100.0%
          pubmed_id   243,749        3.5%
                doi    53,913        0.8%
       article_type        12        0.0%
              title       576        0.0%
            subject        27        0.0%
           keywords 1,693,848       24.6%
copyright_statement   226,399        3.3%
            license    41,453

In [6]:
# ── 2.3 Worst-offender columns across the whole database ────────────────────
worst = (df_missing[df_missing["n_missing"] > 0]
         .sort_values("pct_missing", ascending=False)
         .reset_index(drop=True))
worst["column_ref"] = worst["table"] + "." + worst["column"]

emit()
emit("=" * 60)
emit(f"COLUMNS WITH ANY MISSING DATA: {len(worst)} of {len(df_missing)} total columns")
emit("=" * 60)
view = worst[["column_ref", "total_rows", "n_missing", "pct_missing"]].copy()
view["total_rows"] = view["total_rows"].apply(lambda n: f"{n:,}")
view["n_missing"]  = view["n_missing"].apply(lambda n: f"{n:,}")
view["pct_missing"] = view["pct_missing"].apply(lambda p: f"{p:5.1f}%")
emit(view.to_string(index=False))


COLUMNS WITH ANY MISSING DATA: 46 of 200 total columns
                       column_ref total_rows  n_missing pct_missing
                   article.pmc_id  6,885,547  6,885,547      100.0%
                   journal.abbrev     22,138     22,138      100.0%
               link_rot_run.notes          3          3      100.0%
             link_rot.wayback_url  1,129,798  1,129,798      100.0%
               link_rot.final_url  1,129,798  1,129,791      100.0%
            link_rot.content_type  1,129,798  1,129,791      100.0%
               link_rot.http_code  1,129,798  1,129,791      100.0%
           link_rot.title_snippet  1,129,798  1,129,791      100.0%
supplementary_material.media_type  6,697,313  6,691,477       99.9%
    fuji_assessment.error_message    210,336    209,897       99.8%
       supplementary_material.url  6,697,313  6,677,646       99.7%
                  article.pub_day  6,885,547  6,256,633       90.9%
                     author.email 13,394,080 11,844,568     

---
## 3 · Summary & Exports

In [7]:
# ── 3.1 Headline summary ─────────────────────────────────────────────────────
emit()
emit("=" * 60)
emit("SUMMARY")
emit("=" * 60)
emit(f"Tables audited            : {len(tables)}")
emit(f"Total rows (all tables)   : {df_counts['rows'].sum():,}")
emit(f"Total columns audited     : {len(df_missing)}")
emit(f"Columns with zero missing : {(df_missing['n_missing'] == 0).sum()}")
emit(f"Columns with any missing  : {(df_missing['n_missing'] > 0).sum()}")
if len(worst):
    top = worst.iloc[0]
    emit(f"Most incomplete column    : {top['column_ref']}  ({top['pct_missing']:.1f}% missing)")


SUMMARY
Tables audited            : 28
Total rows (all tables)   : 268,982,485
Total columns audited     : 200
Columns with zero missing : 154
Columns with any missing  : 46
Most incomplete column    : article.pmc_id  (100.0% missing)


In [8]:
# ── 3.2 Write the text report + underlying data ──────────────────────────────
export_dir = Path("../results/fair_db_overview_exports")
export_dir.mkdir(parents=True, exist_ok=True)

txt_path = export_dir / "database_overview.txt"
txt_path.write_text(_report.getvalue())
print(f"✅ {txt_path}  ({len(_report.getvalue()):,} chars)")

df_counts.drop(columns="rows_fmt").to_csv(export_dir / "table_row_counts.csv", index=False)
print(f"✅ {export_dir / 'table_row_counts.csv'}  ({len(df_counts):,} rows)")

df_missing.to_csv(export_dir / "missing_data_audit.csv", index=False)
print(f"✅ {export_dir / 'missing_data_audit.csv'}  ({len(df_missing):,} rows)")

print(f"\nAll outputs saved to ./{export_dir}/")

✅ ../results/fair_db_overview_exports/database_overview.txt  (16,228 chars)
✅ ../results/fair_db_overview_exports/table_row_counts.csv  (28 rows)
✅ ../results/fair_db_overview_exports/missing_data_audit.csv  (200 rows)

All outputs saved to ./../results/fair_db_overview_exports/


In [9]:
con.close()
print("Database connection closed.")

Database connection closed.
